In [1]:

import pandas as pd

csv_path = '/Users/admin/dev/algobetting/infra/data/json/api_football/fixtures_flat.csv'

leagues = [
    'Premier League',
    'League Cup',
    'UEFA Champions League',
    'UEFA Europa Conference League',
    'UEFA Europa League',
    'FA Cup',
    'Community Shield',
    'UEFA Super Cup',
    'FIFA Club World Cup' 
]

df = pd.read_csv(csv_path)
df = df[df['league_name'].isin(leagues)]
df

,fixture_id,date,status,venue,league_id,league_name,league_country,season,round,home_team_id,home_team,away_team_id,away_team,home_goals,away_goals,ht_home,ht_away
102,862929,2022-07-30 17:00:00+01:00,FT,King Power Stadium,528,Community Shield,England,2022,Final,40,Liverpool,50,Manchester City,3.0,1.0,1.0,0.0
112,867946,2022-08-05 20:00:00+01:00,FT,Selhurst Park,39,Premier League,England,2022,Regular Season - 1,52,Crystal Palace,42,Arsenal,0.0,2.0,0.0,1.0
113,867947,2022-08-06 12:30:00+01:00,FT,Craven Cottage,39,Premier League,England,2022,Regular Season - 1,36,Fulham,40,Liverpool,2.0,2.0,1.0,0.0
114,867951,2022-08-06 15:00:00+01:00,FT,St. James' Park,39,Premier League,England,2022,Regular Season - 1,34,Newcastle,65,Nottingham Forest,2.0,0.0,0.0,0.0
116,867949,2022-08-06 15:00:00+01:00,FT,Elland Road,39,Premier League,England,2022,Regular Season - 1,63,Leeds,39,Wolves,2.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2170,1208401,2025-05-25 16:00:00+01:00,FT,Tottenham Hotspur Stadium,39,Premier League,England,2024,Regular Season - 38,47,Tottenham,51,Brighton,1.0,4.0,1.0,0.0
2171,1208397,2025-05-25 16:00:00+01:00,FT,Old Trafford,39,Premier League,England,2024,Regular Season - 38,33,Manchester United,66,Aston Villa,2.0,0.0,0.0,0.0
2172,1208396,2025-05-25 16:00:00+01:00,FT,Anfield,39,Premier League,England,2024,Regular Season - 38,40,Liverpool,52,Crystal Palace,1.0,1.0,0.0,1.0
2173,1208398,2025-05-25 16:00:00+01:00,FT,St. James' Park,39,Premier League,England,2024,Regular Season - 38,34,Newcastle,45,Everton,0.0,1.0,0.0,0.0


In [2]:
# Build a team-centric view: one row per team per fixture
# Stack home and away appearances into a single column

home = df[['fixture_id', 'date', 'season', 'league_name', 'home_team', 'away_team', 'home_goals', 'away_goals']].copy()
home = home.rename(columns={'home_team': 'team', 'away_team': 'opponent', 'home_goals': 'gf', 'away_goals': 'ga'})
home['venue'] = 'home'

away = df[['fixture_id', 'date', 'season', 'league_name', 'away_team', 'home_team', 'away_goals', 'home_goals']].copy()
away = away.rename(columns={'away_team': 'team', 'home_team': 'opponent', 'away_goals': 'gf', 'home_goals': 'ga'})
away['venue'] = 'away'

team_df = pd.concat([home, away], ignore_index=True)
team_df['date'] = pd.to_datetime(team_df['date'], utc=True)
team_df = team_df.sort_values(['team', 'date']).reset_index(drop=True)

print(f"Team-centric rows: {len(team_df)}")
team_df.head()

Team-centric rows: 3274


,fixture_id,date,season,league_name,team,opponent,gf,ga,venue
0,1299349,2024-11-28 17:45:00+00:00,2024,UEFA Europa Conference League,1. FC Heidenheim,Chelsea,0.0,2.0,home
1,1299244,2025-01-23 17:45:00+00:00,2024,UEFA Europa League,1899 Hoffenheim,Tottenham,2.0,3.0,home
2,946891,2022-10-05 19:00:00+00:00,2022,UEFA Champions League,AC Milan,Chelsea,0.0,3.0,away
3,946900,2022-10-11 19:00:00+00:00,2022,UEFA Champions League,AC Milan,Chelsea,0.0,2.0,home
4,971793,2023-02-14 20:00:00+00:00,2022,UEFA Champions League,AC Milan,Tottenham,1.0,0.0,home


In [3]:
# Drop team-seasons with fewer than 38 Premier League fixtures
# (indicates incomplete data — promoted/relegated mid-capture or missing scrape)

pl_counts = (
    team_df[team_df['league_name'] == 'Premier League']
    .groupby(['team', 'season'])
    .size()
    .reset_index(name='pl_fixtures')
)

valid = pl_counts[pl_counts['pl_fixtures'] >= 38][['team', 'season']]
print(f"Valid team-seasons (≥38 PL fixtures): {len(valid)}")
print(f"Dropped: {len(pl_counts) - len(valid)}")

dropped = pl_counts[pl_counts['pl_fixtures'] < 38]
if not dropped.empty:
    print("\nDropped team-seasons:")
    print(dropped.to_string(index=False))

team_df = team_df.merge(valid, on=['team', 'season'], how='inner')
print(f"\nTeam-centric rows after filter: {len(team_df)}")

Valid team-seasons (≥38 PL fixtures): 57
Dropped: 3

Dropped team-seasons:
         team  season  pl_fixtures
      Burnley    2023           34
        Luton    2023           34
Sheffield Utd    2023           34

Team-centric rows after filter: 2750


In [4]:
# Calculate days between games per team (across all competitions)

team_df = team_df.sort_values(['team', 'date']).reset_index(drop=True)

team_df['days_rest'] = (
    team_df.groupby('team')['date']
    .diff()
    .dt.total_seconds()
    .div(86400)
    .round(1)
)

team_df['days_ahead'] = (
    team_df.groupby('team')['date']
    .diff(-1)
    .dt.total_seconds()
    .div(86400)
    .abs()
    .round(1)
)

team_df['day_of_week'] = team_df['date'].dt.day_name()

print(team_df[['team', 'day_of_week', 'date', 'league_name', 'days_rest', 'days_ahead']].head(20).to_string())

       team day_of_week                      date         league_name  days_rest  days_ahead
0   Arsenal      Friday 2022-08-05 19:00:00+00:00      Premier League        NaN         7.8
1   Arsenal    Saturday 2022-08-13 14:00:00+00:00      Premier League        7.8         7.1
2   Arsenal    Saturday 2022-08-20 16:30:00+00:00      Premier League        7.1         7.0
3   Arsenal    Saturday 2022-08-27 16:30:00+00:00      Premier League        7.0         4.1
4   Arsenal   Wednesday 2022-08-31 18:30:00+00:00      Premier League        4.1         3.9
5   Arsenal      Sunday 2022-09-04 15:30:00+00:00      Premier League        3.9         4.1
6   Arsenal    Thursday 2022-09-08 16:45:00+00:00  UEFA Europa League        4.1         9.8
7   Arsenal      Sunday 2022-09-18 11:00:00+00:00      Premier League        9.8        13.0
8   Arsenal    Saturday 2022-10-01 11:30:00+00:00      Premier League       13.0         5.3
9   Arsenal    Thursday 2022-10-06 19:00:00+00:00  UEFA Europa League 

In [5]:
# Categorise rest periods
# short       ≤4 days  — fixture congestion
# normal      5–7 days — standard weekly turnaround
# free_midweek 8–9 days — had a free midweek slot
# long_gap    10+ days — international break, postponement, summer, etc.

team_df['rest_category'] = pd.cut(
    team_df['days_rest'],
    bins=[0, 5, 8, float('inf')],
    labels=['short', 'free_midweek', 'long_gap']
)

print(team_df['rest_category'].value_counts().sort_index())
team_df[['team', 'date', 'league_name', 'days_rest', 'rest_category', 'day_of_week']].head(20)

rest_category
short           1425
free_midweek     894
long_gap         410
Name: count, dtype: int64


,team,date,league_name,days_rest,rest_category,day_of_week
0,Arsenal,2022-08-05 19:00:00+00:00,Premier League,NaN,NaN,Friday
1,Arsenal,2022-08-13 14:00:00+00:00,Premier League,7.8,free_midweek,Saturday
2,Arsenal,2022-08-20 16:30:00+00:00,Premier League,7.1,free_midweek,Saturday
3,Arsenal,2022-08-27 16:30:00+00:00,Premier League,7.0,free_midweek,Saturday
4,Arsenal,2022-08-31 18:30:00+00:00,Premier League,4.1,short,Wednesday
5,Arsenal,2022-09-04 15:30:00+00:00,Premier League,3.9,short,Sunday
6,Arsenal,2022-09-08 16:45:00+00:00,UEFA Europa League,4.1,short,Thursday
7,Arsenal,2022-09-18 11:00:00+00:00,Premier League,9.8,long_gap,Sunday
8,Arsenal,2022-10-01 11:30:00+00:00,Premier League,13.0,long_gap,Saturday
9,Arsenal,2022-10-06 19:00:00+00:00,UEFA Europa League,5.3,free_midweek,Thursday


In [6]:
# Model prep — Premier League games only
# Target: points earned (3 win, 1 draw, 0 loss)
# Features: team rest, opponent rest, home/away

pl = team_df[team_df['league_name'] == 'Premier League'].copy()

pl['result'] = (pl['gf'] > pl['ga']).astype(int) * 3 + (pl['gf'] == pl['ga']).astype(int)

# Self-join on fixture_id to get opponent's rest days
opp_rest = pl[['fixture_id', 'team', 'days_rest', 'rest_category']].copy()
opp_rest.columns = ['fixture_id', 'opponent', 'opp_days_rest', 'opp_rest_category']
pl = pl.merge(opp_rest, on=['fixture_id', 'opponent'], how='left')

pl['rest_advantage'] = pl['days_rest'] - pl['opp_days_rest']

pl_model = pl.dropna(subset=['days_rest', 'opp_days_rest']).copy()
pl_model['is_home'] = (pl_model['venue'] == 'home').astype(int)
pl_model['rest_cat'] = pl_model['rest_category'].astype(str)

print(f"Model rows: {len(pl_model)}")
print(f"\nPoints distribution:\n{pl_model['result'].value_counts().sort_index()}")
print(f"\nRest category distribution:\n{pl_model['rest_cat'].value_counts()}")
pl_model[['team', 'date', 'days_rest', 'opp_days_rest', 'rest_advantage', 'rest_cat', 'result']].head(10)

Model rows: 2042

Points distribution:
result
0    784
1    474
3    784
Name: count, dtype: int64

Rest category distribution:
rest_cat
short           947
free_midweek    748
long_gap        347
Name: count, dtype: int64


,team,date,days_rest,opp_days_rest,rest_advantage,rest_cat,result
1,Arsenal,2022-08-13 14:00:00+00:00,7.8,6.0,1.8,free_midweek,3
2,Arsenal,2022-08-20 16:30:00+00:00,7.1,7.1,0.0,free_midweek,3
3,Arsenal,2022-08-27 16:30:00+00:00,7.0,3.9,3.1,free_midweek,3
4,Arsenal,2022-08-31 18:30:00+00:00,4.1,3.2,0.9,short,3
5,Arsenal,2022-09-04 15:30:00+00:00,3.9,2.9,1.0,short,0
6,Arsenal,2022-09-18 11:00:00+00:00,9.8,14.9,-5.1,long_gap,3
7,Arsenal,2022-10-01 11:30:00+00:00,13.0,13.8,-0.8,long_gap,3
8,Arsenal,2022-10-09 15:30:00+00:00,2.9,4.9,-2.0,short,3
9,Arsenal,2022-10-16 13:00:00+00:00,2.8,7.0,-4.2,short,3
10,Arsenal,2022-10-23 13:00:00+00:00,2.8,3.8,-1.0,short,1


In [7]:
import statsmodels.formula.api as smf

# Model 1: without team fixed effects (baseline)
m1 = smf.ols(
    'result ~ rest_advantage + C(rest_cat, Treatment("short")) + is_home',
    data=pl_model
).fit()

# Model 2: with team fixed effects
m2 = smf.ols(
    'result ~ rest_advantage + C(rest_cat, Treatment("short")) + is_home + C(team)',
    data=pl_model
).fit()

# Compare rest-related coefficients across both models
rest_vars = [c for c in m2.params.index if 'rest' in c or 'home' in c]

print("=== Without team fixed effects ===")
for v in rest_vars:
    if v in m1.params:
        p = m1.pvalues[v]
        sig = '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.1 else ''))
        print(f"  {v:<55} coef={m1.params[v]:+.3f}  p={p:.3f}  {sig}")

print(f"\n=== With team fixed effects ===")
for v in rest_vars:
    p = m2.pvalues[v]
    sig = '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.1 else ''))
    print(f"  {v:<55} coef={m2.params[v]:+.3f}  p={p:.3f}  {sig}")

print(f"\nR² without fixed effects: {m1.rsquared:.3f}")
print(f"R² with    fixed effects: {m2.rsquared:.3f}")

=== Without team fixed effects ===
  C(rest_cat, Treatment("short"))[T.free_midweek]         coef=-0.130  p=0.043  **
  C(rest_cat, Treatment("short"))[T.long_gap]             coef=-0.092  p=0.263  
  rest_advantage                                          coef=-0.005  p=0.005  ***
  is_home                                                 coef=+0.425  p=0.000  ***

=== With team fixed effects ===
  C(rest_cat, Treatment("short"))[T.free_midweek]         coef=+0.034  p=0.597  
  C(rest_cat, Treatment("short"))[T.long_gap]             coef=-0.026  p=0.746  
  C(team)[T.Nottingham Forest]                            coef=-0.929  p=0.000  ***
  rest_advantage                                          coef=-0.003  p=0.074  *
  is_home                                                 coef=+0.419  p=0.000  ***

R² without fixed effects: 0.031
R² with    fixed effects: 0.121
